# 🚲 Beyond Proximity: Spatially Anisotropic Demand Dependencies in Urban Bike-Sharing

> **Weather-Augmented ConvLSTM Analysis · Washington DC Capital Bikeshare · January 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### 📋 Notebook Overview
This notebook implements a **permutation-based spatial dependency analysis** framework for urban bike-sharing demand forecasting using ConvLSTM deep learning, applied to real Washington DC Capital Bikeshare data with weather augmentation.

| | |
|---|---|
| **Study area** | Washington DC, USA |
| **Data** | Capital Bikeshare trips — January 2026 |
| **Grid** | 8×8 spatial cells (~1,500 m each) |
| **Models** | Separate ConvLSTM for member & casual users |
| **Key output** | 64×64 spatial dependency matrix Φ per user type |

**Pipeline:** Data → Weather API → Grid Aggregation → ConvLSTM Training → Φ Matrix → Visualisation

---

## Cell 1: Install Dependencies

In [ ]:
# Install all required libraries
!pip install -q tensorflow shap requests geopandas folium plotly scikit-learn pandas numpy matplotlib seaborn
print('✅ All dependencies installed.')

## Cell 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
import os
import zipfile
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('✅ All libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')

## Cell 3: Configuration

Define the study area bounding box for Washington DC, grid dimensions, temporal resolution, and model hyperparameters.

In [ ]:
class Config:
    # Study area: Washington DC
    STUDY_AREA = {
        'name': 'Washington DC',
        'bbox': {
            'min_lat': 38.80,
            'max_lat': 39.00,
            'min_lon': -77.15,
            'max_lon': -76.90
        }
    }
    GRID_SIZE = 8          # 8x8 = 64 cells
    CELL_SIZE_M = 1500     # ~1,500 m per cell
    TIME_INTERVAL = 30     # minutes per interval
    LOOKBACK_STEPS = 4     # 2 hours of history (4 × 30 min)
    BATCH_SIZE = 32
    EPOCHS = 30
    LEARNING_RATE = 0.001
    DATA_DIR = '/content'
    OUTPUT_DIR = '/content/outputs'

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print(f'📍 Study Area : {config.STUDY_AREA["name"]}')
print(f'📐 Grid       : {config.GRID_SIZE}×{config.GRID_SIZE} = {config.GRID_SIZE**2} cells')
print(f'⏱  Interval   : {config.TIME_INTERVAL} min')
print(f'🔍 Lookback   : {config.LOOKBACK_STEPS} steps ({config.LOOKBACK_STEPS*config.TIME_INTERVAL} min)')
print(f'📁 Output dir : {config.OUTPUT_DIR}')

## Cell 4: Upload & Extract Data

**Instructions:**
1. Download `202601-capitalbikeshare-tripdata.zip` from [Capital Bikeshare System Data](https://capitalbikeshare.com/system-data)
2. Upload to `/content/` using the Colab file panel (left sidebar → Files → Upload)
3. Run this cell

In [ ]:
import shutil

def extract_data(zip_path):
    """Extract Capital Bikeshare trip CSV from zip archive."""
    print(f'📦 Extracting: {zip_path}')
    if not os.path.exists(zip_path):
        print(f'❌ File not found: {zip_path}')
        print('Please upload 202601-capitalbikeshare-tripdata.zip to /content/')
        return None
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(config.DATA_DIR)
    csv_file = os.path.join(config.DATA_DIR, '202601-capitalbikeshare-tripdata.csv')
    if os.path.exists(csv_file):
        print(f'✅ Extracted: {csv_file}')
        return csv_file
    print('❌ CSV not found after extraction'); return None

zip_path = os.path.join(config.DATA_DIR, '202601-capitalbikeshare-tripdata.zip')
csv_file = extract_data(zip_path)

## Cell 5: Load & Preprocess Trip Data

- Parse timestamps, compute trip duration
- Filter invalid trips (< 1 min or > 12 hours)
- Restrict to study area bounding box
- Retain only `member` and `casual` user types

In [ ]:
def load_bikeshare_data(csv_path):
    """Load, clean and spatially filter Capital Bikeshare trip data."""
    print('📥 Loading bike-sharing data...')
    df = pd.read_csv(csv_path)
    print(f'   Raw records : {len(df):,}')
    df['started_at'] = pd.to_datetime(df['started_at'])
    df['ended_at']   = pd.to_datetime(df['ended_at'])
    df['duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60
    df = df[
        (df['duration_min'] >= 1) & (df['duration_min'] <= 720) &
        df['start_lat'].notna() & df['end_lat'].notna() &
        df['member_casual'].isin(['member', 'casual'])
    ].copy()
    print(f'   After cleaning : {len(df):,}  |  Members: {(df.member_casual=="member").sum():,}  |  Casual: {(df.member_casual=="casual").sum():,}')
    b = config.STUDY_AREA['bbox']
    df = df[
        df.start_lat.between(b['min_lat'], b['max_lat']) &
        df.start_lng.between(b['min_lon'], b['max_lon']) &
        df.end_lat.between(b['min_lat'], b['max_lat']) &
        df.end_lng.between(b['min_lon'], b['max_lon'])
    ]
    print(f'   In study area  : {len(df):,}')
    return df

trips_df = load_bikeshare_data(csv_file)
trips_df[['started_at','start_lat','start_lng','end_lat','end_lng','member_casual']].head()

## Cell 6: Fetch Weather Data

Retrieves **hourly** temperature (°C), precipitation (mm), wind speed (km/h), and relative humidity (%) from the [Open-Meteo Historical Archive API](https://open-meteo.com/) — free, no API key required.

Falls back to realistic synthetic January DC values if the API is unreachable.

In [ ]:
def get_weather_data(lat, lon, start_date, end_date):
    """Fetch hourly weather from Open-Meteo archive. Synthetic fallback if unavailable."""
    print('🌤  Fetching weather from Open-Meteo...')
    try:
        resp = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
            'latitude': lat, 'longitude': lon,
            'start_date': start_date, 'end_date': end_date,
            'hourly': ['temperature_2m','precipitation','windspeed_10m','relativehumidity_2m'],
            'timezone': 'America/New_York'
        }, timeout=30)
        d = resp.json()['hourly']
        df = pd.DataFrame({'datetime': pd.to_datetime(d['time']),
            'temperature': d['temperature_2m'], 'precipitation': d['precipitation'],
            'windspeed': d['windspeed_10m'], 'humidity': d['relativehumidity_2m']})
        print(f'✅ {len(df)} hourly records fetched.')
        return df
    except Exception as e:
        print(f'⚠  API unavailable ({e}). Using synthetic January DC weather.')
        dates = pd.date_range(start_date, end_date, freq='H')
        return pd.DataFrame({'datetime': dates,
            'temperature':  np.random.normal(2, 5, len(dates)),
            'precipitation': np.random.exponential(0.3, len(dates)),
            'windspeed':     np.random.gamma(2, 2.5, len(dates)),
            'humidity':      np.random.uniform(50, 85, len(dates))})

weather_df = get_weather_data(
    lat=38.9, lon=-77.03,
    start_date=str(trips_df['started_at'].min().date()),
    end_date=str(trips_df['started_at'].max().date())
)
weather_df.describe().round(2)

## Cell 7: Create Spatial Grid

Divides the DC bounding box into an **8×8 regular grid** of 64 cells.
Each cell spans ≈ 0.025° lat × 0.031° lon ≈ 1,500 m per side.

In [ ]:
def create_grid(bbox, grid_size):
    lat_bins = np.linspace(bbox['min_lat'], bbox['max_lat'], grid_size + 1)
    lon_bins = np.linspace(bbox['min_lon'], bbox['max_lon'], grid_size + 1)
    cells = []
    for i in range(grid_size):
        for j in range(grid_size):
            cells.append({'cell_id': i*grid_size+j+1, 'row': i, 'col': j,
                'min_lat': lat_bins[i], 'max_lat': lat_bins[i+1],
                'min_lon': lon_bins[j], 'max_lon': lon_bins[j+1],
                'center_lat': (lat_bins[i]+lat_bins[i+1])/2,
                'center_lon': (lon_bins[j]+lon_bins[j+1])/2})
    return pd.DataFrame(cells)

grid_df = create_grid(config.STUDY_AREA['bbox'], config.GRID_SIZE)
print(f'📐 Created {len(grid_df)} grid cells')

fig, ax = plt.subplots(figsize=(9, 8))
for _, cell in grid_df.iterrows():
    ax.add_patch(plt.Rectangle((cell.min_lon, cell.min_lat),
        cell.max_lon-cell.min_lon, cell.max_lat-cell.min_lat,
        fill=False, edgecolor='steelblue', lw=0.8))
    ax.text(cell.center_lon, cell.center_lat, str(int(cell.cell_id)),
        ha='center', va='center', fontsize=7, color='firebrick')
ax.set_xlim(config.STUDY_AREA['bbox']['min_lon'], config.STUDY_AREA['bbox']['max_lon'])
ax.set_ylim(config.STUDY_AREA['bbox']['min_lat'], config.STUDY_AREA['bbox']['max_lat'])
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Study Area Grid — Washington DC (8×8)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/grid_layout.png', dpi=300, bbox_inches='tight')
plt.show()

## Cell 8: Aggregate Trips to Grid × Time Intervals

Assigns each trip to its start/end cell, bins by 30-minute intervals,
and computes **pickups** and **dropoffs** per cell per interval per user type.

In [ ]:
def aggregate_to_grid(trips, grid, interval=30):
    print('🔄 Aggregating trips to spatiotemporal grid...')
    trips = trips.copy()
    trips['start_cell'] = -1; trips['end_cell'] = -1
    for _, cell in grid.iterrows():
        sm = (trips.start_lat.between(cell.min_lat, cell.max_lat, inclusive='left') &
              trips.start_lng.between(cell.min_lon, cell.max_lon, inclusive='left'))
        em = (trips.end_lat.between(cell.min_lat, cell.max_lat, inclusive='left') &
              trips.end_lng.between(cell.min_lon, cell.max_lon, inclusive='left'))
        trips.loc[sm, 'start_cell'] = cell.cell_id
        trips.loc[em, 'end_cell']   = cell.cell_id
    trips = trips[(trips.start_cell != -1) & (trips.end_cell != -1)]
    print(f'   Trips within grid: {len(trips):,}')
    trips['time_bin'] = trips['started_at'].dt.floor(f'{interval}min')
    pickups  = trips.groupby(['time_bin','start_cell','member_casual']).size().reset_index(name='pickups')
    dropoffs = trips.groupby(['time_bin','end_cell',  'member_casual']).size().reset_index(name='dropoffs')
    dropoffs = dropoffs.rename(columns={'end_cell':'start_cell'})
    agg = pickups.merge(dropoffs, on=['time_bin','start_cell','member_casual'], how='outer').fillna(0)
    agg = agg.rename(columns={'start_cell':'cell_id'})
    print(f'✅ {len(agg):,} spatiotemporal records created.')
    return agg

aggregated_df = aggregate_to_grid(trips_df, grid_df, config.TIME_INTERVAL)

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, utype, title in zip(axes, ['member','casual'], ['Member Pickups','Casual Pickups']):
    act = aggregated_df[aggregated_df.member_casual==utype].groupby('cell_id')['pickups'].sum()
    mat = grid_df.merge(act, left_on='cell_id', right_index=True, how='left').fillna(0)\
                 .pivot(index='row', columns='col', values='pickups')
    im = ax.imshow(mat, cmap='YlOrRd'); ax.set_title(title, fontweight='bold')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/activity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Cell 9: Prepare Spatiotemporal Sequences

Constructs model inputs:
- **X_spatial**: `[N, 4, 8, 8, 2]` — 4-step lookback of pickup+dropoff grids
- **X_external**: `[N, 7]` — weather (temp, precip, wind, humidity) + temporal (hour, DoW, weekend)
- **y**: `[N, 8, 8, 2]` — target pickup+dropoff grids

In [ ]:
def prepare_training_data(aggregated, weather, grid, user_type='member'):
    print(f'🏗  Preparing sequences for {user_type} users...')
    data = aggregated[aggregated.member_casual == user_type].copy()
    all_times = pd.date_range(data.time_bin.min(), data.time_bin.max(), freq=f'{config.TIME_INTERVAL}min')
    full = pd.MultiIndex.from_product([all_times, range(1, config.GRID_SIZE**2+1)],
                                       names=['time_bin','cell_id']).to_frame(index=False)
    data = full.merge(data, on=['time_bin','cell_id'], how='left')
    data[['pickups','dropoffs']] = data[['pickups','dropoffs']].fillna(0)
    weather['datetime'] = weather['datetime'].dt.floor(f'{config.TIME_INTERVAL}min')
    data = data.merge(weather.rename(columns={'datetime':'time_bin'}), on='time_bin', how='left').ffill().bfill()
    data['hour'] = data.time_bin.dt.hour
    data['day_of_week'] = data.time_bin.dt.dayofweek
    data['is_weekend'] = (data.day_of_week >= 5).astype(int)
    data = data.merge(grid[['cell_id','center_lat','center_lon','row','col']], on='cell_id', how='left')
    timesteps = sorted(data.time_bin.unique())
    G = config.GRID_SIZE; L = config.LOOKBACK_STEPS
    Xs, Xe, ys = [], [], []
    for t in range(L, len(timesteps)):
        hist = []
        for lb in range(L):
            td = data[data.time_bin == timesteps[t-L+lb]].sort_values('cell_id')
            hist.append(np.concatenate([
                td.pickups.values.reshape(G,G,1),
                td.dropoffs.values.reshape(G,G,1)], axis=-1))
        Xs.append(np.stack(hist, axis=0))
        cd = data[data.time_bin == timesteps[t]].iloc[0]
        Xe.append([cd.temperature/30, cd.precipitation, cd.windspeed/20,
                   cd.humidity/100, cd.hour/24, cd.day_of_week/7, cd.is_weekend])
        td = data[data.time_bin == timesteps[t]].sort_values('cell_id')
        ys.append(np.concatenate([td.pickups.values.reshape(G,G,1),
                                   td.dropoffs.values.reshape(G,G,1)], axis=-1))
    X_s, X_e, y = np.array(Xs), np.array(Xe), np.array(ys)
    print(f'✅ Spatial: {X_s.shape} | External: {X_e.shape} | Target: {y.shape}')
    return X_s, X_e, y, data

X_spatial_member, X_external_member, y_member, data_member = prepare_training_data(aggregated_df, weather_df, grid_df, 'member')
X_spatial_casual, X_external_casual, y_casual, data_casual = prepare_training_data(aggregated_df, weather_df, grid_df, 'casual')

## Cell 10: Build ConvLSTM Model

**Architecture:**
- Spatial branch: 3 × ConvLSTM2D (64 filters, 3×3) + BatchNorm + Dropout(0.2)
- External branch: Dense(32) → Dense(64) → spatial broadcast
- Merge → Conv2D(2, 1×1) output layer

Two identical architectures trained independently on member and casual data.

In [ ]:
def build_model(spatial_shape, ext_dim, grid_size):
    si = keras.Input(shape=spatial_shape[1:], name='spatial')
    x = si
    for ret_seq in [True, True, False]:
        x = layers.ConvLSTM2D(64, (3,3), padding='same', return_sequences=ret_seq,
            activation='relu', kernel_regularizer=keras.regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        if ret_seq: x = layers.Dropout(0.2)(x)
    ei = keras.Input(shape=(ext_dim,), name='external')
    e = layers.Dense(32, activation='relu')(ei)
    e = layers.BatchNormalization()(e)
    e = layers.Dense(64, activation='relu')(e)
    e = layers.Reshape((1,1,64))(e)
    e = layers.UpSampling2D(size=(grid_size, grid_size))(e)
    out = layers.Conv2D(2, (1,1), activation='relu')(
          layers.Concatenate()([x, e]))
    model = keras.Model(inputs=[si, ei], outputs=out)
    model.compile(optimizer=keras.optimizers.Adam(config.LEARNING_RATE),
                  loss='mse', metrics=['mae'])
    return model

model_member = build_model(X_spatial_member.shape, X_external_member.shape[1], config.GRID_SIZE)
model_casual = build_model(X_spatial_casual.shape, X_external_casual.shape[1], config.GRID_SIZE)
print('✅ Models built!')
model_member.summary()

## Cell 11: Train Models

- **Split**: 80% train / 20% test (temporal order preserved — no shuffle)
- **Callbacks**: EarlyStopping (patience=10) + ReduceLROnPlateau (factor=0.5, patience=5)
- **Metrics reported**: MAE, RMSE, R²

In [ ]:
def train_model(model, Xs, Xe, y, utype):
    print(f'\n🚂 Training {utype} model...')
    idx = int(0.8 * len(Xs))
    Xstr, Xste = Xs[:idx], Xs[idx:]
    Xetr, Xete = Xe[:idx], Xe[idx:]
    ytr,  yte  = y[:idx],  y[idx:]
    cbs = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
            restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
            patience=5, min_lr=1e-6, verbose=1)
    ]
    hist = model.fit([Xstr, Xetr], ytr,
        validation_data=([Xste, Xete], yte),
        epochs=config.EPOCHS, batch_size=config.BATCH_SIZE,
        callbacks=cbs, verbose=1)
    loss, mae = model.evaluate([Xste, Xete], yte, verbose=0)
    preds = model.predict([Xste, Xete], verbose=0)
    r2 = 1 - np.sum((yte-preds)**2) / np.sum((yte-yte.mean())**2)
    print(f'\n📊 {utype.capitalize()} → MAE: {mae:.3f}  RMSE: {np.sqrt(loss):.3f}  R²: {r2:.3f}')
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, key, lbl in zip(axes, ['loss','mae'], ['Loss (MSE)','MAE']):
        ax.plot(hist.history[key], label='Train')
        ax.plot(hist.history[f'val_{key}'], label='Val')
        ax.set(xlabel='Epoch', ylabel=lbl, title=f'{utype.capitalize()} — {lbl}')
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{config.OUTPUT_DIR}/training_history_{utype}.png', dpi=300, bbox_inches='tight')
    plt.show()
    return hist, (Xste, Xete, yte), preds

history_member, test_member, pred_member = train_model(model_member, X_spatial_member, X_external_member, y_member, 'member')
history_casual, test_casual, pred_casual = train_model(model_casual, X_spatial_casual, X_external_casual, y_casual, 'casual')

## Cell 12: Compute Spatial Dependency Matrix Φ

**Permutation-based perturbation approach** (model-agnostic):

For each (source, target) cell pair:
1. Sample 30 test sequences
2. Zero out source cell values in the lookback window
3. Re-run inference
4. `Φ(src→tgt) = mean |baseline_pred[tgt] − perturbed_pred[tgt]|`

Result: **64×64 directed influence matrix** per user type.

In [ ]:
def compute_phi(model, Xs, Xe, grid_size, n=30):
    print('🧮 Computing Φ matrix (this takes a few minutes)...')
    nc = grid_size ** 2
    phi = np.zeros((nc, nc))
    idx = np.random.choice(len(Xs), min(n, len(Xs)), replace=False)
    Xs_s, Xe_s = Xs[idx], Xe[idx]
    base = model.predict([Xs_s, Xe_s], verbose=0)
    for tgt in range(nc):
        tr, tc = tgt//grid_size, tgt%grid_size
        b_tgt = base[:, tr, tc, :]
        for src in range(nc):
            if src == tgt: continue
            sr, sc = src//grid_size, src%grid_size
            Xp = Xs_s.copy(); Xp[:, :, sr, sc, :] = 0
            pp = model.predict([Xp, Xe_s], verbose=0)
            phi[tgt, src] = np.mean(np.abs(b_tgt - pp[:, tr, tc, :]))
        if (tgt+1) % 16 == 0: print(f'   {tgt+1}/{nc} cells processed')
    print('✅ Φ matrix complete.')
    return phi

phi_member = compute_phi(model_member, test_member[0], test_member[1], config.GRID_SIZE, n=30)
phi_casual = compute_phi(model_casual, test_casual[0], test_casual[1], config.GRID_SIZE, n=30)

print(f'\nMember mean Φ : {phi_member.mean():.5f}  |  max Φ : {phi_member.max():.3f}')
print(f'Casual  mean Φ : {phi_casual.mean():.5f}  |  max Φ : {phi_casual.max():.3f}')
print(f'Ratio (member/casual): {phi_member.mean()/phi_casual.mean():.2f}×')

## Cells 13–18: Visualisation & Export

In [ ]:
# ── Cell 13: Spatial dependency visualisation ───────────────────────────
def plot_phi(phi, grid_df, utype, top_k=10):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    im = axes[0].imshow(phi, cmap='RdBu_r', aspect='auto')
    axes[0].set(title=f'{utype.capitalize()} Spatial Dependency Matrix (Φ)',
                xlabel='Source Cell', ylabel='Target Cell')
    plt.colorbar(im, ax=axes[0], label='Dependency Strength')
    out = phi.sum(axis=0); inn = phi.sum(axis=1)
    gp = grid_df.copy(); gp['net'] = out - inn
    ng = gp.pivot(index='row', columns='col', values='net')
    im2 = axes[1].imshow(ng, cmap='RdBu', aspect='auto')
    axes[1].set(title=f'{utype.capitalize()} Net Spatial Influence',
                xlabel='Column', ylabel='Row')
    plt.colorbar(im2, ax=axes[1], label='Net Influence (Outward − Inward)')
    pairs = sorted([(phi[i,j],i,j) for i in range(len(phi)) for j in range(len(phi)) if i!=j and phi[i,j]>0],
                    reverse=True)[:top_k]
    for v,s,t in pairs:
        axes[1].arrow(s%8, s//8, t%8-s%8, t//8-s//8,
            head_width=0.3, head_length=0.3, fc='green', ec='green',
            alpha=0.7, width=v/phi.max()*0.1)
    plt.tight_layout()
    plt.savefig(f'{config.OUTPUT_DIR}/spatial_dependencies_{utype}.png', dpi=300, bbox_inches='tight')
    plt.show()
    return gp

grid_member_viz = plot_phi(phi_member, grid_df, 'member')
grid_casual_viz = plot_phi(phi_casual, grid_df, 'casual')

# ── Cell 14: Interactive Folium maps ─────────────────────────────────────
def make_folium_map(grid_df, phi, utype):
    out = phi.sum(axis=0); inn = phi.sum(axis=1); net = out - inn
    gd = grid_df.copy(); gd['net'] = net; gd['out'] = out; gd['inn'] = inn
    norm = (net-net.min())/(net.max()-net.min()+1e-9)
    m = folium.Map(location=[gd.center_lat.mean(), gd.center_lon.mean()],
                   zoom_start=11, tiles='CartoDB positron')
    for idx, cell in gd.iterrows():
        v = norm[idx]
        r = 255 if v>0.5 else int(255*v*2)
        b = int(255*(1-(v-0.5)*2)) if v>0.5 else 255
        folium.Rectangle(
            bounds=[[cell.min_lat,cell.min_lon],[cell.max_lat,cell.max_lon]],
            color='#%02x64%02x'%(r,b), fill=True, fillOpacity=0.55, weight=1.5,
            popup=f'<b>Cell {int(cell.cell_id)}</b><br>Net: {cell.net:.3f}<br>Out: {cell.out:.3f}<br>In: {cell.inn:.3f}'
        ).add_to(m)
    m.save(f'{config.OUTPUT_DIR}/spatial_map_{utype}.html')
    print(f'🗺  Saved: spatial_map_{utype}.html')
    return m

map_member = make_folium_map(grid_df, phi_member, 'member')
map_casual = make_folium_map(grid_df, phi_casual, 'casual')

In [ ]:
# ── Cell 15: User-type comparison ────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes[0,0].hist(phi_member.flatten(), bins=50, alpha=0.7, label='Member', color='steelblue', density=True)
axes[0,0].hist(phi_casual.flatten(), bins=50, alpha=0.7, label='Casual', color='darkorange', density=True)
axes[0,0].set(yscale='log', xlabel='Dependency Strength', ylabel='Density',
              title='Distribution of Spatial Dependencies')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

dists, dm, dc = [], [], []
for i in range(64):
    for j in range(64):
        if i!=j:
            d = np.sqrt(((i//8)-(j//8))**2+((i%8)-(j%8))**2)
            dists.append(d); dm.append(phi_member[i,j]); dc.append(phi_casual[i,j])
axes[0,1].scatter(dists, dm, alpha=0.25, s=8, label='Member', color='steelblue')
axes[0,1].scatter(dists, dc, alpha=0.25, s=8, label='Casual', color='darkorange')
axes[0,1].set(xlabel='Grid Distance', ylabel='Dependency Strength',
              title='Spatial Proximity vs. Dependency')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

for ax, phi, title in zip(axes[1], [phi_member, phi_casual], ['Member Outward Influence','Casual Outward Influence']):
    im = ax.imshow(phi.sum(axis=0).reshape(8,8), cmap='YlOrRd')
    ax.set_title(title, fontweight='bold'); plt.colorbar(im, ax=ax)

plt.suptitle('User-Type Spatial Dependency Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{config.OUTPUT_DIR}/user_type_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

r_m = np.corrcoef(dists, dm)[0,1]; r_c = np.corrcoef(dists, dc)[0,1]
print(f'Proximity–dependency r:  Member = {r_m:.3f}  |  Casual = {r_c:.3f}')
print(f'Anisotropy index:  Member = {phi_member.sum(axis=0).std()/phi_member.sum(axis=0).mean():.3f}')
print(f'Member/Casual Φ ratio: {phi_member.mean()/phi_casual.mean():.2f}×')

In [ ]:
# ── Cell 16: Prediction visualisation ───────────────────────────────────
def vis_preds(yt, yp, utype, n=4):
    idx = np.random.choice(len(yt), n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(11, 3.8*n))
    for k, i in enumerate(idx):
        for ax, data, title in zip(axes[k],
            [yt[i,:,:,0], yp[i,:,:,0], np.abs(yt[i,:,:,0]-yp[i,:,:,0])],
            [f'Actual (Sample {k+1})', f'Predicted (Sample {k+1})', f'Abs Error (Sample {k+1})']):
            im = ax.imshow(data, cmap='YlOrRd', vmin=0)
            ax.set_title(title); plt.colorbar(im, ax=ax)
    plt.suptitle(f'{utype.capitalize()} — Prediction Visualisation', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{config.OUTPUT_DIR}/predictions_{utype}.png', dpi=300, bbox_inches='tight')
    plt.show()

vis_preds(test_member[2], pred_member, 'member')
vis_preds(test_casual[2], pred_casual, 'casual')

In [ ]:
# ── Cell 17: Key Findings Summary ────────────────────────────────────────
def r2(model, Xs, Xe, y):
    p = model.predict([Xs, Xe], verbose=0)
    return 1 - np.sum((y-p)**2) / np.sum((y-y.mean())**2)

r2m = r2(model_member, test_member[0], test_member[1], test_member[2])
r2c = r2(model_casual, test_casual[0], test_casual[1], test_casual[2])
anis = phi_member.sum(axis=0).std() / phi_member.sum(axis=0).mean()

print('=' * 65)
print('                  KEY FINDINGS SUMMARY')
print('=' * 65)
print(f'\n MODEL PERFORMANCE')
print(f'   Member R²  : {r2m:.3f}   |   Casual R²  : {r2c:.3f}')
print(f'   Member MAE : {history_member.history["val_mae"][-1]:.3f}   |   Casual MAE : {history_casual.history["val_mae"][-1]:.3f}')
print(f'\n SPATIAL DEPENDENCY')
print(f'   Proximity–dep correlation : {np.corrcoef(dists,dm)[0,1]:.3f}  (WEAK)')
print(f'   Anisotropy index          : {anis:.3f}  (HIGH)')
print(f'\n USER TYPE DIFFERENCES')
print(f'   Member/Casual Φ ratio : {phi_member.mean()/phi_casual.mean():.2f}×')
print(f'   Member mean Φ : {phi_member.mean():.5f}   |   Casual mean Φ : {phi_casual.mean():.5f}')
print('=' * 65)

In [ ]:
# ── Cell 18: Export all results ──────────────────────────────────────────
np.save(f'{config.OUTPUT_DIR}/phi_matrix_member.npy', phi_member)
np.save(f'{config.OUTPUT_DIR}/phi_matrix_casual.npy', phi_casual)

ge = grid_df.copy()
ge['outward_member'] = phi_member.sum(axis=0); ge['inward_member'] = phi_member.sum(axis=1)
ge['net_member']     = ge.outward_member - ge.inward_member
ge['outward_casual'] = phi_casual.sum(axis=0); ge['inward_casual'] = phi_casual.sum(axis=1)
ge['net_casual']     = ge.outward_casual - ge.inward_casual
ge.to_csv(f'{config.OUTPUT_DIR}/spatial_influence_analysis.csv', index=False)

with open(f'{config.OUTPUT_DIR}/summary_statistics.txt','w') as f:
    for k,v in {
        'total_trips': len(trips_df), 'member_trips': (trips_df.member_casual=='member').sum(),
        'casual_trips': (trips_df.member_casual=='casual').sum(), 'grid_cells': 64,
        'member_mean_phi': phi_member.mean(), 'casual_mean_phi': phi_casual.mean(),
        'member_r2': r2m, 'casual_r2': r2c, 'phi_ratio': phi_member.mean()/phi_casual.mean(),
        'anisotropy_member': anis
    }.items(): f.write(f'{k}: {v}\n')

print('✅ All results exported to:', config.OUTPUT_DIR)
print('\nFiles saved:')
for f in sorted(os.listdir(config.OUTPUT_DIR)): print(f'   {f}')
print('\n🎉 Analysis complete!')